<a href="https://colab.research.google.com/github/zwimpee/cursivetransformer/blob/main/debug_hct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clone the cursivetransformer repository and install its requirements
!rm -rf cursivetransformer && git clone https://github.com/zwimpee/cursivetransformer.git
!pip install -r cursivetransformer/requirements.txt

Cloning into 'cursivetransformer'...
remote: Enumerating objects: 2977, done.
remote: Counting objects: 100% (498/498), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 2977 (delta 456), reused 403 (delta 388), pack-reused 2479 (from 1)
Receiving objects: 100% (2977/2977), 64.41 MiB | 9.08 MiB/s, done.
Resolving deltas: 100% (1653/1653), done.
  Cloning https://github.com/callummcdougall/CircuitsVis.git to /tmp/pip-req-build-6a2wpvk9
  Running command git clone --filter=blob:none --quiet https://github.com/callummcdougall/CircuitsVis.git /tmp/pip-req-build-6a2wpvk9
  Resolved https://github.com/callummcdougall/CircuitsVis.git to commit 1e6129d08cae7af9242d9ab5d3ed322dd44b4dd3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which v

In [ ]:
import sys
sys.path.append('/content/cursivetransformer')
from cursivetransformer.model import get_all_args, get_checkpoint, get_latest_checkpoint_artifact
from cursivetransformer.data import create_datasets, offsets_to_strokes, strokes_to_offsets
from cursivetransformer.sample import generate, generate_n_words, plot_strokes
from cursivetransformer.mech_interp import (
    HookedCursiveTransformer,
    HookedCursiveTransformerConfig,
    convert_cursivetransformer_model_config,
    visualize_attention,
    generate_repeated_stroke_tokens,
    generate_random_ascii_context,
    run_and_cache_model_repeated_tokens,
    compute_induction_scores,
    plot_induction_scores,
    plot_head_attention_pattern,
    create_induction_summary,
    ablate_heads,
    get_induction_positions,
    compute_loss_on_induction_positions,
    visualize_attention_patterns
)

import pandas as pd
import os

import copy
import types
from typing import List, Callable, Dict, Optional, Union, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import einops
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import circuitsvis as cv
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from jaxtyping import Float, Int


import transformer_lens.utils as utils
from transformer_lens.hook_points import HookPoint
from transformer_lens import ActivationCache

torch.set_grad_enabled(False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# wandb_api_key - e56bbe426145e5983e72a933299daca195ebb6a7
args = get_all_args(False)
args.sample_only = True
args.max_seq_length = 1250
args.load_from_run_id = '7e9hz1og'
args.wandb_project = 'bigbank_2k'
args.wandb_entity = 'sam-greydanus'
args.dataset_name = 'bigbank'
args.wandb_run_name = 'cursivetransformer_mech_interp_part_3_stroke_attention'
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)

## Verify that the original and hooked versions of the model produce the same output logits for a given sequence

In [ ]:
# Load the original model
original_model, _, _, _, _ = get_checkpoint(args, sample_only=True)
original_model.eval()

In [ ]:
# Load the hooked model
cfg = convert_cursivetransformer_model_config(args)
hooked_model = HookedCursiveTransformer.from_pretrained("cursivetransformer", cfg)
hooked_model = hooked_model.to(device)
hooked_model.eval()

In [ ]:
# Prepare a sample input
sample_tokens = torch.randint(0, args.vocab_size, (1, args.block_size)).to(device)
sample_context = torch.randint(0, args.context_vocab_size, (1, args.context_block_size)).to(device)

In [ ]:
# Run both models
with torch.no_grad():
    original_output = original_model(sample_tokens, sample_context, return_type="logits")
    hooked_output = hooked_model(sample_tokens, sample_context)

In [ ]:
# Compare outputs
max_diff = torch.max(torch.abs(original_output - hooked_output))
print(f"Maximum difference in logits: {max_diff.item()}")

if max_diff > 1e-5:
    print("Models are not equivalent!")
    # If models are not equivalent, let's print more detailed information
    print("Original model output shape:", original_output.shape)
    print("Hooked model output shape:", hooked_output.shape)
    print("Original model output mean:", original_output.mean().item())
    print("Hooked model output mean:", hooked_output.mean().item())
    print("Original model output std:", original_output.std().item())
    print("Hooked model output std:", hooked_output.std().item())
else:
    print("Models are equivalent.")

Maximum difference in logits: 109.5845718383789
Models are not equivalent!
Original model output shape: torch.Size([1, 1250, 382])
Hooked model output shape: torch.Size([1, 1250, 382])
Original model output mean: -14.839642524719238
Hooked model output mean: 0.018848156556487083
Original model output std: 14.006147384643555
Hooked model output std: 0.5719537138938904
